# Fake Estimation for Project 4

This notebook evaluates **all `.pt` checkpoints in the current directory**, ranks them with a mock scoring pipeline, and then displays **11 sample images** for the **top 3 models**.

The evaluation follows the instructor's theory from Lecture 22:

1. **Class coverage / category spread**
2. **Diversity / Inception-style score**
3. **Realism / FID-style score**

This is a practical local estimator, not the official grader.


## Expected inputs

- A local dataset folder such as `synth-cute/`
- A set of diffusion checkpoints in the current directory, for example:
  - `dit_model1_epoch004.pt`
  - `dit_model1_epoch008.pt`
  - `dit_model2_best.pt`
- If `classifier_state_dict.pt` is missing, this notebook will train a classifier and save it.

The notebook does **not** save generated images for every model. It only:

- evaluates all `.pt` files
- ranks them by a combined fake score
- prints the best 3 model names
- displays 11 preview samples for those 3 models


In [ ]:
import pathlib
from collections import OrderedDict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from diffusers import DDPMScheduler, DiTTransformer2DModel
from tqdm.auto import tqdm

from classifier_utils import CLASS_NAMES, JsonLabelDataset, SmallCNNClassifier, train_classifier


In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE =', DEVICE)

CONFIG = {
    'data_root': 'synth-cute',
    'classifier_ckpt': 'classifier_state_dict.pt',
    'num_classes': 11,
    'image_size': 64,
    'in_channels': 3,
    'out_channels': 3,
    'patch_size': 4,
    'num_layers': 8,
    'num_attention_heads': 8,
    'attention_head_dim': 64,
    'num_train_timesteps': 800,
    'num_label_embeddings': 11,
    'uncond_label': 11,
    'sample_seeds_eval': list(range(1, 101)),
    'sample_seeds_preview': list(range(1, 12)),
    'batch_size_eval': 64,
    'classifier_epochs_if_missing': 7,
}

MODEL_DIR = pathlib.Path('.')
MODEL_PATHS = OrderedDict()

for path in sorted(MODEL_DIR.glob('*.pt')):
    if path.name == CONFIG['classifier_ckpt']:
        continue
    MODEL_PATHS[path.stem] = str(path)

print('Found checkpoints:')
for name, path in MODEL_PATHS.items():
    print(name, '->', path)


In [ ]:
def build_dit_model(device=DEVICE):
    model = DiTTransformer2DModel(
        sample_size=CONFIG['image_size'],
        in_channels=CONFIG['in_channels'],
        out_channels=CONFIG['out_channels'],
        patch_size=CONFIG['patch_size'],
        num_layers=CONFIG['num_layers'],
        num_attention_heads=CONFIG['num_attention_heads'],
        attention_head_dim=CONFIG['attention_head_dim'],
        num_embeds_ada_norm=CONFIG['num_label_embeddings'],
    ).to(device)
    return model


def build_scheduler():
    return DDPMScheduler(num_train_timesteps=CONFIG['num_train_timesteps'])


def load_classifier(device=DEVICE):
    ckpt_path = pathlib.Path(CONFIG['classifier_ckpt'])
    classifier = SmallCNNClassifier(in_channels=3, num_classes=len(CLASS_NAMES)).to(device)

    if ckpt_path.exists():
        state_dict = torch.load(ckpt_path, map_location=device)
        classifier.load_state_dict(state_dict)
        classifier.eval()
        print(f'Loaded classifier checkpoint from {ckpt_path.resolve()}')
        return classifier

    print('classifier_state_dict.pt not found. Training classifier now...')
    classifier, best_state_dict = train_classifier(
        data_path=CONFIG['data_root'],
        device=device,
        image_size=CONFIG['image_size'],
        batch_size=64,
        num_epochs=CONFIG['classifier_epochs_if_missing'],
        grayscale=False,
    )
    torch.save(best_state_dict, ckpt_path)
    classifier.load_state_dict(best_state_dict)
    classifier.eval()
    print(f'Saved classifier checkpoint to {ckpt_path.resolve()}')
    return classifier


def classifier_feature_extractor(classifier):
    return classifier.net[:-1]


In [ ]:
def generate_images_for_evaluation(model, scheduler, seeds, device=DEVICE):
    batch_size = len(seeds)

    model.eval()
    scheduler.set_timesteps(CONFIG['num_train_timesteps'])
    scheduler.timesteps = scheduler.timesteps.to(device)

    generators = [torch.Generator(device=device).manual_seed(seed) for seed in seeds]
    samples = torch.stack([
        torch.randn(
            (CONFIG['in_channels'], CONFIG['image_size'], CONFIG['image_size']),
            generator=g,
            device=device,
        )
        for g in generators
    ], dim=0)

    sample_labels = torch.full(
        (batch_size,),
        CONFIG['uncond_label'],
        device=device,
        dtype=torch.long,
    )

    with torch.no_grad():
        for t in tqdm(scheduler.timesteps, leave=False):
            t_batch = torch.full((batch_size,), t.item(), device=device, dtype=torch.long)
            model_output = model(samples, t_batch, class_labels=sample_labels).sample
            samples = scheduler.step(model_output, t, samples).prev_sample

    samples = samples.detach().cpu().clamp(0, 1)
    return samples


In [ ]:
def get_real_dataset_tensor():
    dataset = JsonLabelDataset(CONFIG['data_root'], image_size=CONFIG['image_size'], grayscale=False)
    images = []
    labels = []
    for image, label in dataset:
        images.append(image)
        labels.append(label.item())
    return torch.stack(images), np.array(labels)


def batched_probs_and_features(classifier, images, batch_size=64, device=DEVICE):
    classifier.eval()
    feature_model = classifier_feature_extractor(classifier)

    probs_list = []
    features_list = []
    max_prob_list = []

    with torch.no_grad():
        for start in range(0, len(images), batch_size):
            batch = images[start:start+batch_size].to(device)
            logits = classifier(batch)
            probs = torch.softmax(logits, dim=1)
            features = feature_model(batch)

            probs_list.append(probs.cpu())
            features_list.append(features.cpu())
            max_prob_list.append(probs.max(dim=1).values.cpu())

    probs = torch.cat(probs_list, dim=0).numpy()
    features = torch.cat(features_list, dim=0).numpy()
    max_probs = torch.cat(max_prob_list, dim=0).numpy()
    return probs, features, max_probs


In [ ]:
def compute_inception_score(probs, eps=1e-12):
    py = probs.mean(axis=0, keepdims=True)
    kl = probs * (np.log(probs + eps) - np.log(py + eps))
    kl_per_image = kl.sum(axis=1)
    return float(np.exp(kl_per_image.mean()))


def inception_points(score):
    normalized = np.clip((score - 1.0) / (9.0 - 1.0), 0.0, 1.0)
    return float(normalized * 30.0)


def compute_class_diversity_score(probs):
    mean_probs = probs.mean(axis=0)
    capped_sum = np.minimum(mean_probs, 0.05).sum()
    normalized = float(np.clip(capped_sum / 0.55, 0.0, 1.0))
    points = normalized * 30.0
    low_classes = [CLASS_NAMES[i] for i, p in enumerate(mean_probs) if p < 0.05]
    return mean_probs, normalized, points, low_classes


def covariance(features):
    return np.cov(features, rowvar=False)


def symmetric_matrix_sqrt(mat, eps=1e-8):
    mat = (mat + mat.T) / 2.0
    eigvals, eigvecs = np.linalg.eigh(mat)
    eigvals = np.clip(eigvals, 0.0, None)
    sqrt_diag = np.diag(np.sqrt(eigvals + eps))
    return eigvecs @ sqrt_diag @ eigvecs.T


def compute_mock_fid(real_features, fake_features):
    mu_real = real_features.mean(axis=0)
    mu_fake = fake_features.mean(axis=0)
    sigma_real = covariance(real_features)
    sigma_fake = covariance(fake_features)

    diff = mu_real - mu_fake
    covmean = symmetric_matrix_sqrt(sigma_real @ sigma_fake)
    fid = float(diff @ diff + np.trace(sigma_real + sigma_fake - 2.0 * covmean))
    return max(fid, 0.0)


def fid_points(fid):
    normalized = np.clip((1400.0 - fid) / (1400.0 - 500.0), 0.0, 1.0)
    return float(normalized * 30.0)


In [ ]:
classifier = load_classifier(device=DEVICE)
real_images, real_labels = get_real_dataset_tensor()
real_probs, real_features, real_max_probs = batched_probs_and_features(
    classifier,
    real_images,
    batch_size=CONFIG['batch_size_eval'],
    device=DEVICE,
)

print('Real dataset size:', len(real_images))
print('Real mean classifier confidence:', float(real_max_probs.mean()))


In [ ]:
def evaluate_checkpoint(model_name, checkpoint_path, classifier, real_features):
    model = build_dit_model(device=DEVICE)
    state_dict = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(state_dict)

    scheduler = build_scheduler()

    eval_images = generate_images_for_evaluation(
        model,
        scheduler,
        seeds=CONFIG['sample_seeds_eval'],
        device=DEVICE,
    )

    probs, fake_features, max_probs = batched_probs_and_features(
        classifier,
        eval_images,
        batch_size=CONFIG['batch_size_eval'],
        device=DEVICE,
    )

    is_score = compute_inception_score(probs)
    is_pts = inception_points(is_score)
    mean_probs, class_div_norm, class_div_pts, low_classes = compute_class_diversity_score(probs)
    fid_value = compute_mock_fid(real_features, fake_features)
    fid_pts = fid_points(fid_value)

    predicted_classes = probs.argmax(axis=1)
    predicted_hist = np.bincount(predicted_classes, minlength=len(CLASS_NAMES))

    return {
        'model': model_name,
        'checkpoint_path': checkpoint_path,
        'mean_confidence': float(max_probs.mean()),
        'inception_score': float(is_score),
        'inception_points': float(is_pts),
        'class_diversity_score': float(class_div_norm),
        'class_diversity_points': float(class_div_pts),
        'mock_fid': float(fid_value),
        'fid_points': float(fid_pts),
        'total_fake_points': float(is_pts + class_div_pts + fid_pts),
        'low_classes': ', '.join(low_classes) if low_classes else '(none)',
        'predicted_histogram': predicted_hist,
        'mean_probs': mean_probs,
    }


In [ ]:
results = []
for model_name, checkpoint_path in MODEL_PATHS.items():
    print(f'Evaluating {model_name} ...')
    result = evaluate_checkpoint(model_name, checkpoint_path, classifier, real_features)
    results.append(result)

summary_df = pd.DataFrame([
    {
        'model': r['model'],
        'mean_confidence': r['mean_confidence'],
        'inception_score': r['inception_score'],
        'inception_points': r['inception_points'],
        'class_diversity_score': r['class_diversity_score'],
        'class_diversity_points': r['class_diversity_points'],
        'mock_fid': r['mock_fid'],
        'fid_points': r['fid_points'],
        'total_fake_points': r['total_fake_points'],
        'low_classes': r['low_classes'],
    }
    for r in results
]).sort_values('total_fake_points', ascending=False)

summary_df


In [ ]:
top3 = summary_df.head(3)['model'].tolist()
print('Top 3 models:')
for name in top3:
    print('-', name)


In [ ]:
def show_11_samples_from_checkpoint(model_name, checkpoint_path):
    model = build_dit_model(device=DEVICE)
    state_dict = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(state_dict)

    scheduler = build_scheduler()
    sample_images = generate_images_for_evaluation(
        model,
        scheduler,
        seeds=CONFIG['sample_seeds_preview'],
        device=DEVICE,
    )

    fig, axes = plt.subplots(1, len(CONFIG['sample_seeds_preview']), figsize=(22, 3))
    for i in range(len(CONFIG['sample_seeds_preview'])):
        image = sample_images[i].permute(1, 2, 0).numpy()
        axes[i].imshow(image)
        axes[i].axis('off')
        axes[i].set_title(str(i + 1))
    fig.suptitle(model_name)
    plt.tight_layout()
    plt.show()


top3_results = [r for r in results if r['model'] in top3]
top3_results = sorted(top3_results, key=lambda x: top3.index(x['model']))

for r in top3_results:
    show_11_samples_from_checkpoint(r['model'], r['checkpoint_path'])


## Notes

- This notebook evaluates **all checkpoints first** and only displays **11 samples for the best 3**.
- It uses a mock FID based on your classifier feature space, so treat it as a relative comparison tool.
- The reverse diffusion timestep count here is set to `800` to match your current training notebook.
